# 02_4 Fine-tuning of the direct three-class classifier on synthetic vectors

**Overview.** This notebook repeats the tuning of `02_3_pca_vector_to_classifier_finetunning.ipynb` (same classes, same model, same folds, same search, selection and evaluation) with one change: the classifier is **fitted on synthetic bags** instead of real ones. The synthetic bags are those of `01_3_pca_to_synthetic_vectors.ipynb`: for every class, tile vectors are sampled from cluster-specific Gaussians estimated on the PCA scores of the official Training slides, and every bag blends the size and cluster composition of several real Training slides. With the default method of 01_3, `'hier_lowrank'`, each bag also shifts the mean of every cluster by a bag-specific offset whose variance is the between-slide variance of the real slides (stain, scanner and patient effects shared by all tiles of a slide), so that the bags differ from one another beyond their cluster proportions. The classes are Non-cancer (N, PB, UDH, FEA), non-invasive atypia/neoplasia (ADH and DCIS) and invasive carcinoma (IC). The question is how well a model that has never seen a real tile in its fit data classifies real slides, and how it compares with the models trained on real vectors.

Within every cross-validation fold the model is **fitted on the synthetic slides** (all of them, role Training) and its epoch is chosen on the **real early-stopping slides** of the fold (the same slides as in 02_3), so the early-stopping score measures the transfer to real tissue. The held-out slides of the fold and the official Testing slides are real and are never used for fitting or stopping. With `FRESH_BAGS = True` the synthetic bags are not read from files but **drawn anew in every epoch** from the fitted generators and the composition templates of 01_3 (`FreshSyntheticBags`): the slide names and labels are fixed, the tiles, sizes and cluster mixes change from epoch to epoch, so the model cannot memorise a fixed set of bags. Held-out slides, early-stopping slides and the Testing slides stay real. The number of clusters `syn_k` of the synthetic bags (the folder `k_<K>`) is a hyperparameter of the Bayesian optimisation, together with the dropout, the instance-loss sample size and the tile cap. The search space is narrower than in 02_3: the learning rate, weight decay and weight of the coarse loss are fixed at the values that 02_3 selected (or at the baseline values when no result of 02_3 is available), and the sample size and the tile cap take only the values listed in section 2.

**Scoring and leakage.** The generator of 01_3 saw *all* official Training slides, which are also in the held-out folds of the pooled cross-validation (Training + Validation). The out-of-fold score of a Training slide is therefore optimistic for a model fitted on its synthetic derivatives. The constant `SCORING` selects the score that drives the search: `'cv'` (the default here) pools every held-out slide (optimistic, and comparable only with other synthetic sets from the same generator, not with 02_3, whose models never saw a derivative of a held-out slide), `'validation'` uses only the out-of-fold official Validation slides of patients without Training slides, which the generator never saw (clean, but few slides, hence noisy). The Testing evaluation is unaffected: no Testing slide enters the PCA, the generator or any choice.

**Per-fold PCA and generator (`PER_FOLD`).** With `PER_FOLD = True` the leak is removed at its source. For every cross-validation fold, notebook 01_2 fits the PCA on the fit slides of that fold only and notebook 01_3 fits the clusters, the statistics, the composition templates and the generator on the same slides, so the held-out slides, the early-stopping slides and the Testing slides are never seen by the PCA or by the generator of the fold. In this notebook every fold then uses its own feature space: the model of a fold is fitted on the synthetic bags of that fold, stopped on the early-stopping slides and scored on the held-out slides, all in the PCA space of that fold, and every fold model predicts the Testing slides in its own space (the run carries its data, `predict_ensemble` uses it, and the saved ensemble records the fold and the feature folder of every model). All pooled held-out slides are scored, and `SCORING = 'cv'` is then free of this leakage (the `'validation'` scoring is refused, because it has no purpose). The score is not exactly paired with that of the global-PCA model of 02_3, because the fold models do not share one PCA basis: the difference in a paired comparison combines the effect of the synthetic training and that of the fold-specific PCA. With `PER_FOLD = False` the notebook behaves as before.

```mermaid
flowchart TD
    subgraph IN["Inputs"]
        PCA[("pca_DIMENSION/UNI2-h/<br/>real PCA vectors (01_2)<br/>PER_FOLD: cv_fold_K/pca_DIMENSION/")]
        SYN[("synthetic/METHOD/pca_DIMENSION/k_K/<br/>labels/BRACS_synthetic.csv (01_3)<br/>PER_FOLD: cv_fold_K/synthetic/")]
        LAB[("labels/BRACS.xlsx<br/>class, patient, official set")]
        FULL[("level_1/UNI2-h/<br/>1536-d vectors, Testing slides only")]
        OLD[("classifier/final/<br/>models of 02_1, 02_2, 02_3")]
    end

    subgraph S3["3. Data"]
        D1["3.1 build_slide_table + load_bags<br/>real bags: stop, held-out, Testing"]
        D2["3.2 relabel_table<br/>Non-cancer / atypia-neoplasia / IC"]
        D3["3.3 make_cv_folds<br/>the folds of 02_3"]
        D3b["3.3b load_fold_data<br/>PER_FOLD: bags of every fold space"]
        D4["3.4 load_synthetic_sets<br/>one set per k (PER_FOLD: per fold)"]
        D5["3.5 validation_mask<br/>score of SCORING"]
    end

    subgraph S4["4. Bayesian optimisation"]
        B1["synthetic_objective_factory + run_study<br/>fit on synthetic bags of syn_k,<br/>stop on real stop slides, score on held-out"]
        B2["trials_table + plot_tuning"]
    end

    subgraph S5["5-7. Selection, confirmation, ensemble"]
        C1["select_setting + confirmation<br/>synthetic_cross_validate, 2 repeats"]
        N1["oof_by_repeat + single_vs_ensemble"]
    end

    subgraph S8["8. Testing set, used once"]
        E1["predict_ensemble<br/>real Testing slides"]
        E2["class_views + bootstrap_views"]
        E3["summary_across_sets + paired_bootstrap_views<br/>vs 02_3, 02_1 and 02_2"]
    end

    OUT[("classifier/final/noncancer_noninvasive_invasive_synpcaDIMENSION/<br/>ensemble_date-time.pt")]

    PCA --> D1
    LAB --> D1
    D1 --> D2 --> D3
    SYN --> D4
    D3 --> D3b
    D3b --> B1
    D3 --> D5
    D3 --> B1
    D4 --> B1
    D5 --> B1
    B1 --> B2 --> C1 --> N1 --> E1 --> E2
    E1 --> E3
    OLD --> E3
    FULL --> E3
    N1 --> OUT
```

## 1. Setup

The environment is recorded so the run can be reproduced.

In [ ]:
# STEP 1: Reduce fragmentation - as per NVIDIA/Cuda recommendation
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# STEP 1: Standard-library imports and general-purpose third-party libraries
import sys
import matplotlib
import numpy as np
import optuna
import pandas as pd
import sklearn
import torch

# STEP 2: Print library versions for reproducibility
print('System Version:', sys.version)
print('PyTorch version:', torch.__version__)
print('Scikit-learn version:', sklearn.__version__)
print('Optuna version:', optuna.__version__)
print('Numpy version:', np.__version__)
print('Pandas version:', pd.__version__)
print('Matplotlib version:', matplotlib.__version__)

The model is trained on the GPU when one is available and on the CPU otherwise (`utils.device.get_device`).

In [ ]:
# STEP 1: Select the device (pass e.g. get_device("cpu") to force the CPU)
from utils.device import get_device
DEVICE = get_device()

# STEP 2: Report it
print('Device:', DEVICE, '|', torch.cuda.get_device_name(DEVICE) if DEVICE.type == 'cuda' else 'no GPU, running on the CPU')

The project's own building blocks are imported here. The search, the selection rule and the ensemble file are in `utils/mil_tuning.py`; the synthetic-data folds, the scoring and the objective with the number of clusters are in `utils/synthetic_training.py`; the data, model, cross-validation and metric code is shared with notebooks 02 to 02_3.

In [ ]:
# STEP 1: Standard-library imports
import json
from pathlib import Path

# STEP 2: Import this project's own building blocks
from utils.dataset_overview import build_slide_table
from utils.mil_data import bags_to_device, load_bags
from utils.mil_cv import fold_summary, make_cv_folds
from utils.mil_train import device_description, load_classifier, predict, predict_ensemble, set_seed
from utils.classification_metrics import evaluate_predictions
from utils.final_classifier import (bootstrap_views, class_views, format_differences, format_views, paired_bootstrap_views, plot_final_results, relabel_table, summary_across_sets)
from utils.mil_tuning import (DEFAULT_SPACE, eligible_trials, load_ensemble, oof_by_repeat, plot_tuning, run_study, save_ensemble, score_cv, select_setting, setting_kwargs, single_vs_ensemble,
                              trials_table)
from utils.per_fold import check_fold_metadata, fold_labels_file, fold_synthetic_root, load_fold_data
from utils.synthetic_training import (add_constant_columns, constants_from_02_3, load_synthetic_sets, load_synthetic_sets_per_fold, score_oof, setting_with_k, subset_cv, synthetic_cross_validate, synthetic_objective_factory, synthetic_tile_counts, validation_mask)

## 2. Constants

Every setting is defined here. `DIMENSION` is the number of PCA components: the real vectors are read from `pca_<DIMENSION>/` (notebook 01_2) and the synthetic bags from `SYNTHETIC_ROOT/<SYNTHETIC_METHOD>/pca_<DIMENSION>/k_<K>/` (notebook 01_3, which must have been run with that dimension and with every `K_VALUES` entry); the label sheet of the synthetic slides is `SYNTHETIC_LABELS_FILE`. `CACHE_IN_RAM = True` keeps the synthetic bags that were read in memory as float32 (about tiles x `DIMENSION` x 4 bytes per bag, at most 8 GB per set, the least recently used first dropped), because the bags are read in every epoch of every trial; the default `False` memory-maps the files instead. `BAGS_ON_GPU = True` (CUDA only) keeps the real bags and the synthetic bags that were read in GPU memory as float32, so that a batch is gathered, padded and standardised on the GPU without a host-to-GPU copy; the synthetic bags of all `K_VALUES` together then occupy up to `GPU_CACHE_GB` of GPU memory (least recently used dropped first, across the sets) and the real bags about 6 GB at 256 components, so lower `GPU_CACHE_GB` if the GPU runs out of memory. `FRESH_BAGS` switches between the saved bags (`False`) and bags that are drawn in every epoch (`True`; `CACHE_IN_RAM` then has no effect, and the study and the ensemble folder get the suffix `_fresh`, so the two kinds are never mixed). Fresh bags need the generators and the composition templates that 01_3 stores in `cluster_models.pt`; a folder written by an earlier version of 01_3 lacks the templates and must be generated again with `OVERWRITE = True` (same seed, same saved bags). Because every epoch sees new bags, the best epoch is expected later than with fixed bags, so `EPOCHS` and `PATIENCE` may need to be raised, and drawing 700 bags per epoch and fold adds CPU time to every trial. `FEATURES_FULL_DIR` holds the original 1536-dimensional vectors, read only for the Testing slides of the comparison with notebooks 02_1 and 02_2.

`PER_FOLD` (default `False`) selects the per-fold PCA spaces and generators (see the overview): it needs notebook 01_2 and notebook 01_3 run with their own `PER_FOLD = True` and the same `N_FOLDS`, `SEED` (the CV seed, 0) and `EARLY_STOP_FRACTION`, which gives `FEATURES_ROOT/cv_fold_<k>/pca_<DIMENSION>/` and `FEATURES_ROOT/cv_fold_<k>/synthetic/`; the folds of this notebook are checked against those that 01_2 recorded. All folds' real bags are held in memory (about five times the real bags of 02_3); `LAZY_FOLDS = True` reads a fold when it is first used and keeps only one in memory, which costs time for each repeat and for the Testing prediction. With `PER_FOLD` the real bags of the global PCA are not moved to the GPU (they are read only for the table and for the comparison of section 8.3), and the synthetic cache `GPU_CACHE_GB` is divided among the folds. `SCORING` is `'cv'` or `'validation'` (see the overview); `PER_FOLD` needs `'cv'`. The searched parameters and ranges are `SPACE`: the dropout (continuous), the instance-loss sample size `k_sample` (16 or 32), the tile cap `max_tiles` (8192 or no cap) and `syn_k`, the number of clusters of the synthetic bags. `DEFAULT_SPACE` of `utils/mil_tuning.py` is not changed, so the other notebooks keep their space. The learning rate, weight decay and weight of the coarse loss are read by `constants_from_02_3` from the setting that 02_3 selected (the config of its latest ensemble file, else the best row of its trials table, else the values in `DEFAULT_CONSTANTS`); the printed line states the source of each value. the first trial is `BASELINE` (the middle value of `K_VALUES` and moderate values of the other parameters). Class weighting, tile dropout, the weight of the slide loss, the number of epochs and the patience are fixed as in 02_3, and so are the recall floor `MIN_CLASS_RECALL`, `N_TRIALS`, `N_CONFIRM` and `REPEATS`. The study is named after the dimension and the scoring, so studies for different dimensions and scorings are kept apart.

`SYNTHETIC_METHOD = 'hier_lowrank'` reads the bags that 01_3 wrote with the hierarchical generator; the earlier set of the `'gaussian'` method is untouched and can be selected by changing this constant. `SCORING = 'cv'` scores every held-out slide. The generator saw all official Training slides, so this score is optimistic; it can be compared with the score of the `'gaussian'` set under the same scoring, but not with the cross-validation score of 02_3. The clean subset of section 7 is the check that is not affected by this leakage.

**Cost of the tile cap.** With `max_tiles = None` every epoch reads all tiles of every bag; the batch size is then 2 (`batch_size_for`), real bags have up to about 35,000 tiles, and a trial takes several times as long as with a cap of 8192 (batch size 4). With `CACHE_IN_RAM = True` the synthetic bags that were read are held in memory (tiles x `DIMENSION` x 4 bytes per bag) in a least-recently-used cache of at most 8 GB for all `K_VALUES` together; the default `False` reads the memory-mapped files and leaves the caching to the operating system.


In [ ]:
# Major variables
DIMENSION = 256                                                         # STEP 1: Number of PCA components (needs pca_<DIMENSION>/ from 01_2 and the synthetic bags of 01_3 for that dimension)
FEATURES_ROOT = Path('/mnt/Scratch/phd_streamembed_features')           # STEP 2: Folder with level_1/ (original vectors), the pca_<d>/ folders and synthetic/
FEATURES_DIR = FEATURES_ROOT / f'pca_{DIMENSION}'                       # STEP 3: Real PCA vectors (stop, held-out and Testing slides)
FEATURES_FULL_DIR = FEATURES_ROOT / 'level_1'                           # STEP 4: Original 1536-d vectors (only the Testing slides are read, for the comparisons of section 8)
SYNTHETIC_ROOT = FEATURES_ROOT / 'synthetic'                            # STEP 5: Root of the synthetic bags of notebook 01_3
SYNTHETIC_METHOD = 'hier_lowrank_shared'                                # STEP 6: Generation method (folder name; 'gaussian' is the earlier set)
SYNTHETIC_LABELS_FILE = Path('labels/BRACS_synthetic.csv')              # STEP 7: Label sheet of the synthetic slides (written by 01_3)
K_VALUES = [100, 150, 250]                                              # STEP 8: Numbers of clusters of the synthetic sets that the search can choose from
SCORING = 'cv'                                                          # STEP 9: 'cv' (all held-out slides; optimistic, not comparable with 02_3) or 'validation' (held-out official Validation slides unseen by the generator)
PER_FOLD, LAZY_FOLDS = True, True                                       # STEP 9b: True: one PCA and one synthetic set per CV fold (01_2 and 01_3 with PER_FOLD); True for LAZY_FOLDS reads one fold's bags at a time (less memory, slower)
CACHE_IN_RAM = False                                                    # STEP 10: True keeps the synthetic bags that were read in memory (float32, at most 8 GB for all k together, least recently used dropped first); False memory-maps the files
BAGS_ON_GPU = True                                                      # STEP 10b: Keep the real bags and the synthetic bags that were read in GPU memory (float32), so batches need no host-to-GPU copy (ignored on the CPU)
GPU_CACHE_GB = 16                                                       # STEP 10c: GPU memory for the synthetic bags of all k together, least recently used dropped first (about 6 GB for the real bags come on top; lower it if the GPU runs out of memory)
FRESH_BAGS = False                                                      # STEP 11: True draws new synthetic bags in every epoch from the generators of 01_3 (needs its 'composition' key); False reads the saved bags
MODEL_NAME = 'UNI2-h'                                                   # STEP 12: Subfolder with the per-slide features
LABELS_FILE = Path('labels/BRACS.xlsx')                                 # STEP 13: Slide labels, patients and official split
MODEL_DIR = Path('classifier/final')                                    # STEP 14: Folder of the saved classifiers
SAVE_MODELS = True                                                      # STEP 15: Save the ensemble as ensemble_<date-time>.pt
SEED = 0                                                                # STEP 16: Seed of every random choice
N_FOLDS, EARLY_STOP_FRACTION = 5, 0.2                                   # STEP 17: Outer folds; share of the non-held-out slides used to pick the epoch (patient-grouped); as in 02_3
EPOCHS, PATIENCE = 100, 5                                               # STEP 18: Maximum epochs; early-stopping patience (counted after the warm-up)
WEIGHTING, TILE_DROPOUT, BAG_WEIGHT = 'sqrt', 0.5, 0.7                  # STEP 19: Class weighting, share of tiles removed per training bag, weight of the slide loss in CLAM-MB
N_TRIALS = 20                                                           # STEP 20: Trials of the Bayesian optimisation (finished trials; a restart continues the study)
STUDY_DIR = Path('outputs/02_4_tuning')                                 # STEP 21: Folder of the study (SQLite), the trials table
STUDY_NAME = f'three_class_syn_pca{DIMENSION}_{SCORING}' + ('_fresh' if FRESH_BAGS else '') + ('_perfold' if PER_FOLD else '')   # STEP 22: Name of the study inside the database (one study per dimension, scoring and kind of bags)
MIN_CLASS_RECALL = 0.40                                                 # STEP 23: Smallest out-of-fold recall that every class must have (a float, a dict per class, or None)
N_CONFIRM, REPEATS = 2, 2                                               # STEP 24: Settings confirmed with new seeds; repeats per fold (models of the ensemble = folds x repeats)
N_BOOT = 1000                                                           # STEP 25: Bootstrap resamples (patients) for the intervals
THREE_MAPPING = {'N': 'Non-cancer', 'PB': 'Non-cancer', 'UDH': 'Non-cancer', 'FEA': 'Non-cancer',
                 'ADH': 'Non-invasive atypia/neoplasia', 'DCIS': 'Non-invasive atypia/neoplasia', 'IC': 'Invasive carcinoma'}   # STEP 26: BRACS class -> group (order = model outputs)
TIERS_3 = {'non-cancer': ['Non-cancer'], 'lesion': ['Non-invasive atypia/neoplasia', 'Invasive carcinoma']}                      # STEP 27: Tiers of the coarse auxiliary loss
DEFAULT_CONSTANTS = dict(lr=2e-4, weight_decay=1e-1, coarse_weight=0.0)  # STEP 28: Values that 02_3 did not provide (the setting of notebook 02_1)
CONSTANTS, CONSTANTS_SOURCE = constants_from_02_3(MODEL_DIR / f'noncancer_noninvasive_invasive_pca{DIMENSION}', Path('outputs/02_3_tuning') / f'trials_three_class_pca{DIMENSION}.csv', DEFAULT_CONSTANTS)   # STEP 29: lr, weight decay and coarse-loss weight fixed at the 02_3 values
SPACE = dict(dropout=DEFAULT_SPACE['dropout'], k_sample=[16, 32], max_tiles=[8192, None], syn_k=K_VALUES)   # STEP 30: Search ranges (this notebook only): (low, high) numeric, list = categorical; syn_k = clusters of the synthetic bags
BASELINE = dict(dropout=0.25, k_sample=16, max_tiles=8192, syn_k=K_VALUES[len(K_VALUES) // 2])   # STEP 31: First trial (searched parameters only)
FIXED = dict(epochs=EPOCHS, patience=PATIENCE, tile_dropout=TILE_DROPOUT, bag_weight=BAG_WEIGHT, weighted_val_loss=True, tiers=TIERS_3, **CONSTANTS)   # STEP 32: train_model arguments that are not searched
print(CONSTANTS_SOURCE)
assert not PER_FOLD or SCORING == 'cv', "PER_FOLD scores every held-out slide: set SCORING = 'cv'"
STUDY_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)

## 3. Data

### 3.1 Real slides and tile embeddings

The real bags (PCA scores of the UNI2-h CLS tokens, `DIMENSION` components, from notebook 01_2) are loaded once. They serve as the early-stopping, held-out and Testing slides; the slide table holds the official split of BRACS. The dimension is checked against `DIMENSION`.

In [ ]:
# STEP 1: Label table with the official split and the bags of tile embeddings (float32)
table = build_slide_table(LABELS_FILE)
data = load_bags(FEATURES_DIR, MODEL_NAME, table)
if BAGS_ON_GPU and not PER_FOLD:                           # with PER_FOLD the bags of the folds go to the GPU instead (section 3.3b)
    data['bags'] = bags_to_device(data['bags'], DEVICE)     # stays in GPU memory; the batches are padded and standardised there

# STEP 2: The vectors must have the chosen dimension
assert data['dim'] == DIMENSION, f"the files in {FEATURES_DIR} have {data['dim']} components, not {DIMENSION}"
print('PCA dimension:', data['dim'])

### 3.2 The three classes

Non-cancer groups N, PB, UDH and FEA; ADH and DCIS form one class; IC keeps its own class. The real slides per class and part are listed.

In [ ]:
# STEP 1: Relabel the table and the bags for the three classes
table3, classes3 = relabel_table(data['table'], THREE_MAPPING)
data3 = {**data, 'table': table3, 'classes': classes3}

# STEP 2: Slides per class and part
pd.crosstab(table3['label'], table3['set']).reindex(classes3)[['Training', 'Validation', 'Testing']]

### 3.3 Cross-validation folds

The official Training and Validation slides are pooled and divided into the same patient-grouped, class-stratified folds as in notebook 02_3 (same `N_FOLDS`, `EARLY_STOP_FRACTION` and `SEED`), so the held-out slides, the early-stopping slides and the Testing slides are identical and the results are paired with those of 02_3. The Testing slides are in no fold. With `PER_FOLD = True` the same folds are the ones whose fit slides fitted the PCA and the generator of each fold; their slide lists are compared with those stored by notebook 01_2 and any difference stops the run. In this notebook the *fit* slides of a fold are not used: the model is fitted on synthetic slides, and the early-stopping slides of the fold select the epoch.

In [ ]:
# STEP 1: The folds (the Testing slides are left out of every fold)
folds = make_cv_folds(table3, N_FOLDS, SEED, EARLY_STOP_FRACTION)

# STEP 2: Slides and patients per part, and the held-out slides per class
fold_summary(folds, classes3)

### 3.3b Per-fold feature spaces

With `PER_FOLD = True` the real bags of every fold are read from the PCA files of that fold (`cv_fold_<k>/pca_<DIMENSION>/`, notebook 01_2), which hold the pool slides and the official Testing slides transformed by the PCA fitted on the fit slides of the fold only. The dimension is checked for every fold. Each fold keeps the folder it was read from (`features_dir`), which the saved ensemble records. Without `PER_FOLD` nothing is read here.

In [ ]:
# STEP 1: The folds must be those that 01_2 used for the PCA of every fold; then read the bags of each fold's own space
if PER_FOLD:
    check_fold_metadata(folds, FEATURES_ROOT, DIMENSION)
    FOLD_DATA = load_fold_data(FEATURES_ROOT, N_FOLDS, DIMENSION, MODEL_NAME, table, DEVICE if BAGS_ON_GPU else None, LAZY_FOLDS)
    print('Per-fold PCA spaces:', [d['features_dir'] if not LAZY_FOLDS else f'fold {k} (lazy)' for k, d in enumerate(FOLD_DATA)][:2], '...')
else:
    FOLD_DATA = None

### 3.4 Synthetic bags

The synthetic slides of the method `SYNTHETIC_METHOD` and of every `k` in `K_VALUES` are read from the label sheet of notebook 01_3 and given the three-class labels of `THREE_MAPPING`. Every file is checked for its dimension and a few for finite values; no synthetic slide may carry a real slide identifier. In the fit data, each synthetic slide is a separate unit (no patient), and the class weights and the standardisation of the model come from these slides only. With `FRESH_BAGS = True` the label sheet still defines the slides, but no file is read: each access to a bag draws a new one from the generator and the templates of its BRACS class, with a random generator seeded by the seed, the epoch and the index of the slide (the same epoch and seed give the same bag), and the tile counts in the table are those of the saved set. The feature scaler is fitted once on a fixed draw of at most 100 evenly spaced bags (epoch 0), not on every epoch's bags, so the standardisation does not drift during training. With `PER_FOLD = True` the sets of every fold are read from that fold's folder and label sheet (`labels/BRACS_synthetic_cv_fold_<k>.csv`, written by 01_3), so the model of a fold is fitted on bags of a generator that only saw the fit slides of that fold; the tables describe fold 0. The class balance is set by the generator (the same number of synthetic bags per BRACS class), so it differs from the real Training set; the class-weighted loss of 02_3 is retained. The tables give the synthetic slides per class and the tiles per slide for each `k`.

In [ ]:
GPU_SYNTHETIC = BAGS_ON_GPU and DEVICE.type != 'cpu'
# STEP 1: The synthetic bags of every k (read in training, or drawn anew in every epoch when FRESH_BAGS), with the three-class labels
if PER_FOLD:
    synthetic_sets = load_synthetic_sets_per_fold([fold_labels_file(SYNTHETIC_LABELS_FILE, k) for k in range(N_FOLDS)], [fold_synthetic_root(FEATURES_ROOT, k) for k in range(N_FOLDS)], SYNTHETIC_METHOD, DIMENSION, K_VALUES, THREE_MAPPING, classes3,
                                                    cache=CACHE_IN_RAM or GPU_SYNTHETIC, fresh=FRESH_BAGS, seed=SEED, max_cache_gb=(GPU_CACHE_GB if GPU_SYNTHETIC else 8.0) / N_FOLDS, device=DEVICE if GPU_SYNTHETIC else None)
    shown_sets = {k: v[0] for k, v in synthetic_sets.items()}        # tables of fold 0
else:
    synthetic_sets = load_synthetic_sets(SYNTHETIC_LABELS_FILE, SYNTHETIC_ROOT, SYNTHETIC_METHOD, DIMENSION, K_VALUES, THREE_MAPPING, classes3, CACHE_IN_RAM or GPU_SYNTHETIC, fresh=FRESH_BAGS, seed=SEED, max_cache_gb=GPU_CACHE_GB if GPU_SYNTHETIC else 8.0, device=DEVICE if GPU_SYNTHETIC else None)
    shown_sets = synthetic_sets

# STEP 2: Synthetic slides per class, and tiles per synthetic slide
display(pd.DataFrame({k: s['table']['label'].value_counts().reindex(classes3) for k, s in shown_sets.items()}).rename_axis('slides per class', axis=1))
display(pd.DataFrame({k: synthetic_tile_counts(s).describe() for k, s in shown_sets.items()}).T.round(0).rename_axis('tiles per slide'))

### 3.5 Slides of the score

With `SCORING = 'cv'` every held-out slide of the pool is scored. With `'validation'` only the official Validation slides of patients that have no Training slide are scored, because the generator of notebook 01_3 was estimated on all official Training slides (`utils.synthetic_training.validation_mask`). The number of scored slides per class is listed.
With `PER_FOLD = True` the generator of a fold never saw its held-out slides, so the pooled score is free of that leak and the clean subset below is only a subset. Without it, because the generator saw all official Training slides, the pooled score (`'cv'`) is optimistic and comparable only with synthetic sets scored in the same way, not with the cross-validation score of 02_3; the clean subset is kept for the check of section 7.


In [ ]:
# STEP 1: The pool slides (order of the out-of-fold arrays) and the scored ones
pool_slides = folds[0]['table']['slide'].tolist()
SCORE_MASK = validation_mask(table3, pool_slides) if SCORING == 'validation' else np.ones(len(pool_slides), dtype=bool)
VALIDATION_MASK = validation_mask(table3, pool_slides)                  # always kept: the clean subset, for the comparison of the two scorings in section 7

# STEP 2: Scored slides per class, for the chosen scoring and for the clean subset
labels_pool = folds[0]['table']['label'].to_numpy()
pd.DataFrame({'scored (SCORING)': pd.Series(labels_pool[SCORE_MASK]).value_counts(), 'clean subset': pd.Series(labels_pool[VALIDATION_MASK]).value_counts()}).reindex(classes3).fillna(0).astype(int)

## 4. Bayesian optimisation

One trial trains `N_FOLDS` models, one per fold, with a sampled setting and a sampled number of clusters `syn_k`: each model is fitted on all synthetic slides of that `k`, its epoch is selected on the real early-stopping slides of its fold, and it predicts the real held-out slides of the fold. The value of the trial is the macro-averaged F1 of the out-of-fold predictions of the slides chosen by `SCORING`. The optimiser is the tree-structured Parzen estimator of optuna (Bergstra et al., 2011); the study is stored in SQLite, so a restart continues it.

In [ ]:
# STEP 1: The objective (one trial = one cross-validation on the synthetic bags of the sampled k) and the study
objective = synthetic_objective_factory(data3, folds, synthetic_sets, DEVICE, FIXED, WEIGHTING, SEED, SPACE, SCORING, SCORE_MASK, FRESH_BAGS, FOLD_DATA)
study = run_study(STUDY_NAME, f'sqlite:///{STUDY_DIR / "study.db"}', objective, N_TRIALS, SEED, BASELINE)

### 4.1 Trials

Every finished trial with its parameters (including `syn_k`), the out-of-fold macro-F1 of the scored slides, the out-of-fold recall and F1 of each class, the standard deviation over folds and the minutes it took. The learning rate, weight decay and weight of the coarse loss are constant (`CONSTANTS`) and are added as columns so that the selection rule and the later sections can read them. Trial 0 is `BASELINE`.

In [ ]:
# STEP 1: The trials table, best first (ties: lower log-loss), with the constant parameters added
trials = add_constant_columns(trials_table(study), CONSTANTS)
trials.to_csv(STUDY_DIR / f'trials_{STUDY_NAME}.csv', index=False)
trials.round(3)

### 4.2 Optimisation history and importance

In [ ]:
# STEP 1: Figure
fig_tuning = plot_tuning(study)

### 4.3 Number of clusters

The best and the median macro-F1 of the trials for each `syn_k`. The trials differ in their other parameters as well, so the table shows association, not the effect of the number of clusters alone.

In [ ]:
# STEP 1: Macro-F1 of the trials by number of clusters of the synthetic bags
trials.groupby('syn_k')['macro_f1'].agg(trials='size', best='max', median='median').round(3)

### 4.3 Best recall per class

The highest out-of-fold recall that any trial reached for each class. A recall floor above these values would remove every trial.

In [ ]:
# STEP 1: Best out-of-fold recall of each class over the trials
trials[[f'recall_{c}' for c in classes3]].max().round(3).to_frame('best recall').T

## 5. Selection

The final setting is chosen without the Testing set, in three steps: settings in which some class has an out-of-fold recall below `MIN_CLASS_RECALL` are removed; the best pooled macro-F1 is found among the rest; and among the settings within one standard error of it (the standard error of the fold macro-F1 values of the scored slides), the one with the higher dropout, then the higher weight decay is taken (one-standard-error rule, Hastie et al., 2009). Macro-F1 gives every class the same weight, which suits the smaller classes; sensitivity alone is not used, since it is maximised by assigning every slide to the class.

In [ ]:
# STEP 1: Chosen setting and the candidates within one standard error of the best
chosen, candidates = select_setting(trials, MIN_CLASS_RECALL, classes3)
display(candidates.round(3))
print('Chosen: trial', int(chosen['trial']), setting_with_k(chosen))

## 6. Confirmation with new seeds

The chosen setting and the next best settings are trained again with `REPEATS` repeats per fold and seeds that the search did not use, because the best value of many trials is optimistic. The same selection rule is then applied to these scores, and the final setting is the one it chooses.

In [ ]:
# STEP 1: The settings to confirm: the chosen one, then the best remaining ones
order = [int(chosen['trial'])] + [int(t) for t in eligible_trials(trials, MIN_CLASS_RECALL, classes3)['trial'] if int(t) != int(chosen['trial'])]
confirm_trials = order[:N_CONFIRM]

# STEP 2: Cross-validation of each with REPEATS repeats and new seeds, on the synthetic bags of its k
cvs, rows = {}, []
for number in confirm_trials:
    setting = setting_with_k(trials[trials['trial'] == number].iloc[0])
    cvs[number] = synthetic_cross_validate(data3, folds, synthetic_sets[setting['syn_k']], DEVICE, WEIGHTING, REPEATS, SEED + 5000, FRESH_BAGS, FOLD_DATA, **setting_kwargs(setting, FIXED))
    rows.append({'trial': number, **setting, **score_oof(cvs[number], classes3, SCORING, SCORE_MASK)})
confirmation = pd.DataFrame(rows).sort_values(['macro_f1', 'log_loss'], ascending=[False, True]).reset_index(drop=True)
confirmation.round(3)

The same selection rule is applied to the confirmation scores, which come from `REPEATS` models per fold, and gives the final setting and its fold models.

In [ ]:
# STEP 1: The final setting by the same rule
final, final_candidates = select_setting(confirmation, MIN_CLASS_RECALL, classes3)
final_trial, final_setting = int(final['trial']), setting_with_k(final)
final_cv = cvs[final_trial]
print('Final: trial', final_trial, final_setting)

## 7. Ensemble

The ensemble consists of the models of the final setting only: one per fold and repeat, `N_FOLDS` x `REPEATS` models, each trained on a different patient-grouped part of Training + Validation and stopped at its own epoch. A held-out slide is predicted by the models of its own fold and their probabilities are averaged, which gives the out-of-fold estimate below without the Testing set; on the Testing set all models predict every slide. With `PER_FOLD = True` each model predicts in the PCA space of its own fold. Models of different settings are not mixed, because the same folds ranked the settings. The table compares each single repeat with the average of the repeats; the average over the confirmed settings is shown for comparison. The tables are restricted to the slides of the chosen `SCORING`; the last table gives the out-of-fold macro-F1 of the ensemble for all held-out slides and for the clean subset that the generator never saw, which shows how far the pooled score is optimistic.
With `SCORING = 'cv'` the pooled score is optimistic (the generator saw all official Training slides) and is comparable only with other synthetic sets scored in the same way; the clean-subset row of the last table is the check.


In [ ]:
# STEP 1: Single repeats against the ensemble, on the scored out-of-fold slides
per_repeat = [p[SCORE_MASK] for p in oof_by_repeat(final_cv, data3, folds, DEVICE)]
scored_cv = subset_cv(final_cv, classes3, SCORE_MASK)
display(single_vs_ensemble(scored_cv, per_repeat, classes3).round(3))

# STEP 2: For comparison, the average over the confirmed settings (the same folds ranked them, so this is optimistic)
from sklearn.metrics import f1_score
mixed = np.mean([cvs[t]['proba'] for t in confirm_trials], axis=0)[SCORE_MASK]
print('Macro F1 of the ensemble of the final setting:', round(score_cv(scored_cv, classes3)['macro_f1'], 3),
      '| of the average over the confirmed settings:', round(f1_score(scored_cv['y'], mixed.argmax(1), average='macro'), 3))

# STEP 3: The two scorings of the final ensemble
display(pd.DataFrame({'all held-out slides (cv)': score_oof(final_cv, classes3, 'cv'), 'clean subset (validation)': score_oof(final_cv, classes3, 'validation', VALIDATION_MASK)}).loc[['macro_f1', 'log_loss'] + [f'recall_{c}' for c in classes3]].round(3).T)

## 8. Evaluation on the Testing set

The Testing labels are read here for the first time. Every model of the ensemble predicts every Testing slide (with `PER_FOLD = True`, in the PCA space of its own fold) and the probabilities are averaged; the class with the highest probability is the prediction. For each class the measures are computed one class against the rest (for the macro row, the multi-class accuracy and the unweighted mean of the others), with 95% intervals from a bootstrap that resamples patients.

### 8.1 Confusion matrix, ROC curves and measures

In [ ]:
# STEP 1: Testing slides, in table order, and the ensemble probabilities
test = table3[table3['set'] == 'Testing'].reset_index(drop=True)
y_test = np.array([classes3.index(l) for l in test['label']])
patients_test = test['patient'].to_numpy()
proba_test = predict_ensemble(final_cv['runs'], data3, test['slide'].tolist(), DEVICE)['proba']

# STEP 2: Measures with patient-bootstrap intervals, figure and table
NAME = f'Synthetic-trained PCA-{DIMENSION} 3-class ensemble (02_4)'
evaluation = evaluate_predictions(y_test, proba_test, classes3)
views = {NAME: class_views(y_test, proba_test, classes3)}
intervals = {NAME: bootstrap_views(y_test, proba_test, patients_test, classes3, N_BOOT, SEED)}
fig_test = plot_final_results(NAME, evaluation['confusion'], y_test, proba_test, classes3)
display(format_views(views[NAME], intervals[NAME]))

### 8.2 Original vectors of the Testing slides

The comparison models of notebooks 02_1 and 02_2 were trained on the 1536-dimensional vectors, so they must score the original vectors of the Testing slides; only these slides are read at full dimension. The slides are the same as above, in the same order.

In [ ]:
# STEP 1: The original CLS vectors of the Testing slides, with the three-class labels
full = load_bags(FEATURES_FULL_DIR, MODEL_NAME, data['table'][data['table']['set'] == 'Testing'])
full_table3, _ = relabel_table(full['table'], THREE_MAPPING)
data_full3 = {**full, 'table': full_table3, 'classes': classes3}

# STEP 2: Same slides, same order, as the PCA ensemble predicts
assert full['table']['slide'].tolist() == test['slide'].tolist(), 'the Testing slides differ between the PCA and the original vectors'
print('Original vector dimension:', full['dim'])

### 8.3 Comparison with the PCA ensemble of notebook 02_3

With `PER_FOLD = True` the two ensembles do not share one PCA basis (02_3 uses the global PCA), so the paired difference combines the effect of the synthetic training with that of the fold-specific PCA; the comparison is not exactly paired in the sense of a single changed factor.

The ensemble saved by notebook 02_3 (`classifier/final/noncancer_noninvasive_invasive_pca<DIMENSION>/`) was tuned and trained with the same classes, folds and search on the **real** PCA vectors of the same dimension. Both ensembles predict the same Testing slides (the PCA vectors) and the difference (synthetic-trained minus real-trained) comes with a 95% interval from a paired patient bootstrap, in which both models are scored on the same resamples. An interval that contains zero is not distinguishable from no change at this sample size (87 slides, 31 patients). The comparison isolates the effect of replacing the real fit slides by synthetic ones; the two searches are separate, and 02_4 also searches the number of clusters.

In [ ]:
# STEP 1: The ensemble of notebook 02_3 (the latest file of its folder), on the same slides and the same PCA vectors
pca_files = sorted((MODEL_DIR / f'noncancer_noninvasive_invasive_pca{DIMENSION}').glob('ensemble_*.pt'))
if not pca_files:
    print('No ensemble in', MODEL_DIR / f'noncancer_noninvasive_invasive_pca{DIMENSION}', ': run notebook 02_3 with this DIMENSION first. Comparison skipped.')
else:
    pca_ens = load_ensemble(pca_files[-1], DEVICE)
    assert pca_ens['classes'] == classes3, 'The two ensembles must have the same classes in the same order'
    proba_pca = predict_ensemble(pca_ens['runs'], data3, test['slide'].tolist(), DEVICE)['proba']

    # STEP 2: Measures of both, and the paired bootstrap of the difference
    views[f'Real PCA-{DIMENSION} ensemble (02_3)'] = class_views(y_test, proba_pca, classes3)
    display(summary_across_sets(views).round(3))
    differences = paired_bootstrap_views(y_test, proba_test, proba_test.argmax(1), proba_pca, proba_pca.argmax(1), patients_test, classes3, N_BOOT, SEED)
    display(format_differences(differences))

### 8.4 Comparison with the direct three-class model of notebook 02_1

The model saved by section 10 of notebook 02_1 (`classifier/final/noncancer_adhdcis_ic/`, classes Non-cancer, ADH&DCIS and IC, the same slides in the same groups; trained on the 1536-dimensional vectors) predicts the same Testing slides. The difference (synthetic-trained ensemble minus that model) comes with a 95% interval from a paired patient bootstrap, in which both models are scored on the same resamples. An interval that contains zero is not distinguishable from no change at this sample size (87 slides, 31 patients).

In [ ]:
# STEP 1: The previous direct three-class model (the latest file of its folder), on the same slides
previous_files = sorted((MODEL_DIR / 'noncancer_adhdcis_ic').glob('classifier_model_*.pt'))
if not previous_files:
    print('No classifier in', MODEL_DIR / 'noncancer_adhdcis_ic', ': run section 10 of notebook 02_1 first. Comparison skipped.')
else:
    previous = load_classifier(previous_files[-1], DEVICE)
    assert len(previous['classes']) == len(classes3), 'The two models must have the same number of classes'
    assert previous['classes'][0] == 'Non-cancer' and previous['classes'][-1] == 'IC', 'The class order of the two models must be the same (the group names differ)'
    proba_previous = predict(previous['model'], data_full3, test['slide'].tolist(), DEVICE, previous['scaler'])['proba']

    # STEP 2: Measures of both, and the paired bootstrap of the difference
    views['Direct 3-class, 1536-d (02_1)'] = class_views(y_test, proba_previous, classes3)
    display(summary_across_sets(views).round(3))
    differences = paired_bootstrap_views(y_test, proba_test, proba_test.argmax(1), proba_previous, proba_previous.argmax(1), patients_test, classes3, N_BOOT, SEED)
    display(format_differences(differences))

### 8.5 Comparison with the tuned ensemble of notebook 02_2

The ensemble saved by notebook 02_2 (`classifier/final/noncancer_noninvasive_invasive_tuned/`) uses the same classes, folds and search, but is trained on the 1536-dimensional vectors. Both ensembles predict the same Testing slides, and the difference (synthetic-trained PCA minus real-trained 1536-d) comes with a 95% interval from the paired patient bootstrap. The comparison combines the effect of the dimension reduction and of the synthetic fit data, together with the setting that each search chose; the searches are separate, so it does not isolate either factor.

In [ ]:
# STEP 1: The ensemble of notebook 02_2 (the latest file of its folder), on the same slides
tuned_files = sorted((MODEL_DIR / 'noncancer_noninvasive_invasive_tuned').glob('ensemble_*.pt'))
if not tuned_files:
    print('No ensemble in', MODEL_DIR / 'noncancer_noninvasive_invasive_tuned', ': run notebook 02_2 first. Comparison skipped.')
else:
    tuned = load_ensemble(tuned_files[-1], DEVICE)
    assert tuned['classes'] == classes3, 'The two ensembles must have the same classes in the same order'
    proba_tuned = predict_ensemble(tuned['runs'], data_full3, test['slide'].tolist(), DEVICE)['proba']

    # STEP 2: Measures of both, and the paired bootstrap of the difference
    views['Tuned, 1536-d (02_2)'] = class_views(y_test, proba_tuned, classes3)
    display(summary_across_sets(views).round(3))
    differences = paired_bootstrap_views(y_test, proba_test, proba_test.argmax(1), proba_tuned, proba_tuned.argmax(1), patients_test, classes3, N_BOOT, SEED)
    display(format_differences(differences))

### 8.6 Summary of the compared results

All models on the same Testing slides: the macro row of each model (accuracy, sensitivity, specificity, precision, F1 and AUROC; one class against the rest, then averaged without weights). Differences with intervals are given in sections 8.3 to 8.5; the Testing set has few slides and patients, so small differences are not resolved.

In [ ]:
# STEP 1: The measures of every model that was available
display(summary_across_sets(views).round(3))

## 9. Save

The ensemble is written to one file (the folder name gets the suffix `_perfold` with `PER_FOLD`, and every model records its fold and feature folder; `utils.per_fold.attach_fold_data` gives a loaded ensemble the bags of its folds) with the weights, scaler and settings of every model (`utils.mil_tuning.save_ensemble`; `load_ensemble` returns the models for `predict_ensemble`). The configuration records the searched ranges, the final setting (including the number of clusters), the scoring, the study the kind of synthetic bags (`fresh`) and the metadata of the synthetic set that the ensemble was fitted on.

In [ ]:
# STEP 1: What is needed to reproduce and to reuse the ensemble
chosen_sets = synthetic_sets[final_setting['syn_k']]
syn_folders = [s['folder'] for s in chosen_sets] if PER_FOLD else [chosen_sets['folder']]
syn_folder = syn_folders[0]
syn_metadata = json.loads((syn_folder / 'metadata.json').read_text()) if (syn_folder / 'metadata.json').exists() else {}
config = dict(
    pca=dict(dimension=DIMENSION, vectors=str(FEATURES_DIR / MODEL_NAME), metadata=str(FEATURES_DIR / 'metadata.json')),
    synthetic=dict(root=str(SYNTHETIC_ROOT), method=SYNTHETIC_METHOD, labels_file=str(SYNTHETIC_LABELS_FILE), k_values=K_VALUES, syn_k=final_setting['syn_k'], fresh=FRESH_BAGS, folder=str(syn_folder), folders=[str(f) for f in syn_folders], per_fold=PER_FOLD, metadata=syn_metadata),
    scoring=SCORING, seed=SEED, n_folds=N_FOLDS, repeats=REPEATS, early_stop_fraction=EARLY_STOP_FRACTION, weighting=WEIGHTING, fixed={k: v for k, v in FIXED.items() if k != 'tiers'}, tiers=TIERS_3,
    final_trial=final_trial, final_setting=final_setting, space={k: list(v) if isinstance(v, (tuple, list)) else v for k, v in SPACE.items()}, n_trials=N_TRIALS, min_class_recall=MIN_CLASS_RECALL,
    mapping=THREE_MAPPING, classes=classes3, features_dir=str(FEATURES_DIR), per_fold=PER_FOLD, fold_features_dirs=[r['features_dir'] for r in final_cv['runs'][:N_FOLDS]] if PER_FOLD else None, model_name=MODEL_NAME, tile_params=data['tile_params'], device=device_description(DEVICE),
    versions=dict(torch=torch.__version__, sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__, optuna=optuna.__version__), dropped_slides=data['missing'],
)

# STEP 2: Save the models of the ensemble
if SAVE_MODELS:
    path = save_ensemble(final_cv['runs'], MODEL_DIR / (f'noncancer_noninvasive_invasive_synpca{DIMENSION}' + ('_fresh' if FRESH_BAGS else '') + ('_perfold' if PER_FOLD else '')), config)
    print('Ensemble of', len(final_cv['runs']), 'models saved to', path.resolve())
else:
    print('SAVE_MODELS is False: nothing was written.')